# TUMSOEV MuseTalk 1.5 — ручной бесплатный тест

Этот чистый ноутбук не содержит чужих фото, видео или голосов. Вы сами загружаете одно видео и одну аудиодорожку. Первый тест автоматически ограничен **5 секундами**.

Перед запуском: **Runtime → Change runtime type → T4 GPU**, затем **Runtime → Run all**.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
assert torch.cuda.is_available(), 'Включите T4 GPU: Runtime → Change runtime type → T4 GPU'

## 1. Установка официального MuseTalk 1.5
Первая установка и загрузка весов может занять заметное время.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git-lfs
!rm -rf /content/MuseTalk
!git clone -q https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk
%cd /content/MuseTalk
!pip -q install 'numpy<2' edge-tts openmim
!pip -q install -r requirements.txt
!mim install -q mmengine
!mim install -q 'mmcv==2.0.1'
!mim install -q 'mmdet==3.1.0'
!mim install -q 'mmpose==1.1.0'
!bash ./download_weights.sh
print('MuseTalk готов')

## 2. Загрузите два файла
Выберите **одно видео** (MP4/MOV/WEBM) и **одно аудио** (WAV/MP3/M4A). Используйте только материалы, на которые у вас есть разрешение.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, os, shutil

os.makedirs('/content/tumsoev_lipsync', exist_ok=True)
uploaded = files.upload()
video_ext = {'.mp4','.mov','.webm','.mkv','.avi'}
audio_ext = {'.wav','.mp3','.m4a','.aac','.flac','.ogg'}
videos = [n for n in uploaded if Path(n).suffix.lower() in video_ext]
audios = [n for n in uploaded if Path(n).suffix.lower() in audio_ext]
assert videos and audios, 'Нужно выбрать одно видео и одно аудио.'
source_video = '/content/' + Path(videos[0]).name
source_audio = '/content/' + Path(audios[0]).name
Path(source_video).write_bytes(uploaded[videos[0]])
Path(source_audio).write_bytes(uploaded[audios[0]])
video = '/content/tumsoev_lipsync/input_5s_25fps.mp4'
audio = '/content/tumsoev_lipsync/voice_5s.wav'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',source_video,'-t','5','-vf','scale=min(720\,iw):-2,fps=25,format=yuv420p','-an','-c:v','libx264','-crf','18',video], check=True)
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',source_audio,'-t','5','-af','apad,atrim=0:5','-ar','16000','-ac','1',audio], check=True)
print('Файлы подготовлены:', video, audio)

## 3. Запустите lip‑sync
`bbox_shift=-2` — спокойный старт. Если рот слишком слабый или широкий, попробуйте -4, 0, 2 или 4 и повторите только эту ячейку.

In [ ]:
from pathlib import Path
import glob, shutil
cfg = 'task_0:\n  video_path: /content/tumsoev_lipsync/input_5s_25fps.mp4\n  audio_path: /content/tumsoev_lipsync/voice_5s.wav\n'
Path('/content/MuseTalk/configs/inference/tumsoev_manual.yaml').write_text(cfg)
%cd /content/MuseTalk
!rm -rf /content/MuseTalk/results/tumsoev_manual
!python -m scripts.inference --inference_config configs/inference/tumsoev_manual.yaml --result_dir results/tumsoev_manual --unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json --version v15 --bbox_shift -2
outs = glob.glob('/content/MuseTalk/results/tumsoev_manual/**/*.mp4', recursive=True)
assert outs, 'MuseTalk не создал MP4 — прочитайте ошибку выше.'
OUT = '/content/TUMSOEV_MUSETALK_5s.mp4'
shutil.copy2(outs[0], OUT)
print('ГОТОВО:', OUT)

In [ ]:
from IPython.display import Video, display
from google.colab import files
display(Video(OUT, embed=True))
files.download(OUT)